In [3]:
!pip install dotenv openai
#!pip uninstall transformers -y
#!pip install transformers

  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached openai-2.15.0-py3-none-any.whl.metadata (29 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached jiter-0.12.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.2 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
Using cached openai-2.15.0-py3-none-any.whl (1.1 MB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached jiter-0.12.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86

# Preparing for API connection

In [1]:
# get api api key, source: https://www.geeksforgeeks.org/python/how-to-create-and-use-env-files-in-python/

# importing os module for environment variables
import os
# importing necessary functions from dotenv library
from dotenv import load_dotenv, dotenv_values 
# loading variables from .env file
load_dotenv(dotenv_path="/dss/dsshome1/03/ge87wod2/.env",override=True) 

# accessing and printing value
#print(os.getenv("GEMINI_API_KEY"))
OPENROUTER_API_KEY=os.getenv("OPENROUTER_API_KEY")

import requests
import json
response = requests.get(
  url="https://openrouter.ai/api/v1/key",
  headers={
    "Authorization": f"Bearer {OPENROUTER_API_KEY}"
  }
)
print(json.dumps(response.json(), indent=2))


{
  "data": {
    "label": "sk-or-v1-25a...d92",
    "is_provisioning_key": false,
    "limit": 18,
    "limit_reset": null,
    "limit_remaining": 17.309535729,
    "include_byok_in_limit": false,
    "usage": 0.690464271,
    "usage_daily": 0,
    "usage_weekly": 0,
    "usage_monthly": 0.34838265,
    "byok_usage": 0,
    "byok_usage_daily": 0,
    "byok_usage_weekly": 0,
    "byok_usage_monthly": 0,
    "is_free_tier": false,
    "expires_at": null,
    "rate_limit": {
      "requests": -1,
      "interval": "10s",
      "note": "This field is deprecated and safe to ignore."
    }
  }
}


In [ ]:
# credits before one language
{
  "data": {
    "label": "sk-or-v1-25a...d92",
    "is_provisioning_key": false,
    "limit": 18,
    "limit_reset": null,
    "limit_remaining": 17.838582579,
    "include_byok_in_limit": false,
    "usage": 0.161417421,
    "usage_daily": 0.0385767,
    "usage_weekly": 0.0385767,
    "usage_monthly": 0.161417421,
    "byok_usage": 0,
    "byok_usage_daily": 0,
    "byok_usage_weekly": 0,
    "byok_usage_monthly": 0,
    "is_free_tier": false,
    "expires_at": null,
    "rate_limit": {
      "requests": -1,
      "interval": "10s",
      "note": "This field is deprecated and safe to ignore."
    }
  }
}
# credits after one language
{
  "data": {
    "label": "sk-or-v1-25a...d92",
    "is_provisioning_key": false,
    "limit": 18,
    "limit_reset": null,
    "limit_remaining": 17.804664679,
    "include_byok_in_limit": false,
    "usage": 0.195335321,
    "usage_daily": 0.0724946,
    "usage_weekly": 0.0724946,
    "usage_monthly": 0.195335321,
    "byok_usage": 0,
    "byok_usage_daily": 0,
    "byok_usage_weekly": 0,
    "byok_usage_monthly": 0,
    "is_free_tier": false,
    "expires_at": null,
    "rate_limit": {
      "requests": -1,
      "interval": "10s",
      "note": "This field is deprecated and safe to ignore."
    }
  }
}

# Zero-Shot Prompting

In [ ]:
model = "google/gemini-2.5-flash"
model = "openai/gpt-oss-120b"
model_name = model.split("/")[-1]
sigmorphon_path = "/dss/dsshome1/03/ge87wod2/morphological-inflection/2023InflectionST/part1/data/"
dataset = ".tst"
data_path = "./data/" + model_name + "/"
lang_code = "deu"
lang = "German"

from openai import OpenAI
client = OpenAI(
		base_url="https://openrouter.ai/api/v1",
		api_key=OPENROUTER_API_KEY,
)
in_path = sigmorphon_path + lang_code + dataset
out_path = data_path + lang_code + "_" + model_name + ".out"
import os
if not os.path.isdir(data_path): os.mkdir(data_path)
with open (in_path,"r") as in_file, open(out_path,"w") as out_file:
				correct = 0
				lines = in_file.readlines()
				line_count = len(lines)
				for line in lines:
					if line == "": continue
					lemma, features, target = line.split("\t")
					while (True):
						response = client.chat.completions.create(
								extra_headers={
								},
								model=model,
								messages=[
										{"role": "system", "content": f"""You are an expert documentary linguist specializing in morphological inflection in the language {lang}. You are given a lemma in {lang} and features in the hierarchical annotation schema UniMorph 4.0. Inflect the lemma according to the features and output only the inflected form."""},
										{"role": "user", "content": f"Inflect lemma + features: {lemma} + {features}"}
								],
						)
						try: 
							pred = (response.choices[0].message).content.strip()
							target = target.strip()
							if pred == target: 
										correct = correct + 1
										print (pred,"is correct.")
							else: print(pred,target)
							out_file.write(lemma + "\t" + features + "\t" + pred + "\n")
							out_file.flush()
							break
						except:	
							print("retry")
				acc = correct / line_count
				out_file.write("test acc "+lang+" " + str(acc)+"\n")	
					
print(acc,lang)

Aikido is correct.
Aikido is correct.
Aikidos is correct.
Aikido is correct.
Anreisen is correct.
Anreise is correct.
Anreisen is correct.
Anreise is correct.
